# Albumentations를 활용한 이미지 데이터 전처리 및 변환

```python
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_transform = A.Compose([
    A.Resize(64, 64),
    A.HorizontalFlip(p=0.5),#image probability 20%
    #이미지의 좌우 방향성에 대한 의존도를 줄여 모델이 다양한 방향의 데이터를 학습할 수 있도록 돕는다. 
    A.Rotate(limit=45, p=0.5),  #-5 ~ 5 rotate
    #이미지가 약간 회전된 상황에서도 모델이 일관된 성능을 발휘할 수 있도록 학습 
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, always_apply=False, p=1.0),
    ToTensorV2()
])

test_transform = A.Compose([
    A.Resize(64, 64),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225), max_pixel_value=255.0, always_apply=False, p=1.0),
    ToTensorV2()
])
```

## Dataset 설정 

```python
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, img_paths, labels=None, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        if self.labels is not None:
            label = self.labels[idx]
            return image, label
        else:
            return image
```


```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(128 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = x.view(-1, 128 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 모델 생성
num_classes = len(train_df['label'].unique())
model = SimpleCNN(num_classes)
print(model)
```

네, 거의 맞습니다. `fc2`의 출력이 `num_classes`인 이유는 각 클래스에 대응하는 점수 하나씩을 만들기 위해서입니다.

예를 들어 클래스가 다음 3개라면:

```python
classes = ["고양이", "강아지", "새"]
```

마지막 층은 샘플마다 3개의 값을 출력합니다.

```python
self.fc2 = nn.Linear(256, 3)

# 출력 예시
tensor([[2.1, 0.3, -1.2]])
```

각 위치는 클래스별 예측 점수입니다.

```text
고양이:  2.1
강아지:  0.3
새:    -1.2
```

다만 중요한 점은 `fc2`의 출력이 확률 자체는 아니라는 것입니다. 이 값을 로짓(logit), 즉 정규화되지 않은 클래스 점수라고 부릅니다.

확률이 필요하면 `softmax`를 적용합니다.

```python
logits = model(images)
probabilities = F.softmax(logits, dim=1)
```

예를 들어:

```python
logits        = [2.1, 0.3, -1.2]
probabilities = [0.83, 0.14, 0.03]
```

이 확률들의 합은 1입니다. 최종 예측 클래스는 가장 큰 로짓 또는 확률의 인덱스로 구합니다.

```python
predicted_class = logits.argmax(dim=1)
```

로짓에 `argmax`를 적용하든 softmax 결과에 `argmax`를 적용하든 예측 클래스는 같습니다.

학습할 때 `nn.CrossEntropyLoss()`를 사용한다면 모델의 `forward()`에서는 softmax를 적용하지 않는 것이 맞습니다.

```python
criterion = nn.CrossEntropyLoss()

logits = model(images)        # shape: [batch_size, num_classes]
loss = criterion(logits, labels)
```

`CrossEntropyLoss` 내부에서 로짓을 확률적으로 처리하는 `log_softmax` 연산이 포함되기 때문입니다. 모델에서 softmax까지 적용하면 연산이 중복되고 학습이 제대로 되지 않을 수 있습니다.

전체적인 출력 크기 변화는 다음과 같습니다.

```text
fc1 출력: [batch_size, 256]
                     ↓
fc2 출력: [batch_size, num_classes]
                     ↓
각 행이 이미지 한 장의 클래스별 점수
```

예를 들어 배치에 이미지가 32장이고 클래스가 10개라면:

```python
output = model(images)
print(output.shape)
# torch.Size([32, 10])
```

즉 `output[i][j]`는 `i`번째 이미지가 `j`번째 클래스일 점수입니다.

참고로 코드의 생성자는 실제 Python 코드에서 반드시 밑줄 두 개를 사용해야 합니다.

```python
def __init__(self, num_classes):
    super().__init__()
```

정리하면:

- 단일 정답 다중 클래스 분류에서는 마지막 출력 수가 `num_classes`
- 각 출력값은 확률이 아니라 클래스별 로짓
- 추론 시 `softmax`를 적용하면 클래스별 확률
- 학습 시 `CrossEntropyLoss`를 사용한다면 모델에 softmax를 넣지 않음
- 가장 큰 값을 가진 출력의 인덱스가 예측 클래스

따라서 질문하신 “마지막 출력층에 클래스별 예측 확률이 들어간다”는 이해는 방향상 맞지만, 정확히는 “클래스별 점수가 나오고, softmax를 적용하면 클래스별 확률이 된다”라고 보면 됩니다.

# 모델 학습 및 검증 함수 구현 


```python
# 학습 함수 정의
def train(model, criterion, optimizer, train_loader):
    model.train()
    running_loss = 0.0
    corrects = 0
    for images, labels in train_loader:
        images, labels = images.float(), labels.long()  # 데이터 타입 변환
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        corrects += torch.sum(preds == labels.data)
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = corrects.double() / len(train_loader.dataset)
    return epoch_loss, epoch_acc
```

이 코드에서 `criterion`이 일반적으로 `nn.CrossEntropyLoss()`라고 가정하면, loss와 accuracy는 다음과 같이 계산됩니다.

## 1. Loss 계산 공식

모델 출력은 확률이 아니라 클래스별 로짓입니다.

```python
outputs = model(images)
```

배치 크기가 $ B $, 클래스 수가 $ C $라면:

$$
\text{outputs.shape} = [B, C]
$$

샘플 \(i\)에 대한 클래스 \(j\)의 로짓을 \(z_{i,j}\)라고 할 때, softmax 확률은:

$$
p_{i,j}
=
\frac{e^{z_{i,j}}}
{\sum_{k=1}^{C} e^{z_{i,k}}}
$$

정답 클래스가 \(y_i\)라면 해당 샘플의 Cross Entropy Loss는:

$$
L_i = -\log(p_{i,y_i})
$$

배치의 평균 loss는:

$$
L_{\text{batch}}
=
-\frac{1}{B}
\sum_{i=1}^{B}
\log(p_{i,y_i})
$$

PyTorch에서는 다음 코드가 softmax와 cross entropy를 한꺼번에 처리합니다.

```python
criterion = nn.CrossEntropyLoss()
loss = criterion(outputs, labels)
```

따라서 학습할 때는 모델 출력에 직접 `softmax`를 적용하지 않습니다.

### 간단한 예시

정답이 클래스 0이라고 해보겠습니다.

```python
logits = [2.0, 1.0, 0.1]
softmax(logits) ≈ [0.659, 0.242, 0.099]
```

정답 클래스의 확률은 `0.659`이므로 loss는:

$$
-\log(0.659) \approx 0.417
$$

정답 클래스의 확률이 높을수록 loss가 작아집니다.

---

## 2. 현재 코드의 epoch loss

현재 코드는 각 배치의 평균 loss를 더합니다.

```python
running_loss += loss.item()
```

마지막에 배치 개수로 나눕니다.

```python
epoch_loss = running_loss / len(train_loader)
```

수식으로 표현하면:

$$
L_{\text{epoch}}
=
\frac{1}{M}
\sum_{m=1}^{M}
L_{\text{batch},m}
$$

여기서 \(M\)은 전체 배치 개수입니다.

모든 배치의 크기가 동일하다면 전체 샘플의 평균 loss와 같습니다. 하지만 마지막 배치는 크기가 작을 수 있기 때문에 엄밀하게는 약간의 오차가 생길 수 있습니다.

예를 들어:

```text
1번 배치: 32개
2번 배치: 32개
3번 배치: 4개
```

현재 코드는 4개짜리 마지막 배치도 32개짜리 배치와 동일한 비중으로 평균을 냅니다.

---

## 3. Accuracy 계산 공식

다음 코드는 각 샘플에서 가장 큰 로짓을 가진 클래스를 선택합니다.

```python
_, preds = torch.max(outputs, 1)
```

더 간단하게는 다음과 같이 작성할 수 있습니다.

```python
preds = outputs.argmax(dim=1)
```

정답과 예측값을 비교합니다.

```python
preds == labels
```

예를 들어:

```python
preds  = [0, 2, 1, 1]
labels = [0, 1, 1, 1]
```

비교 결과는:

```python
[True, False, True, True]
```

따라서 맞힌 개수는 3개이고 accuracy는:

$$
\text{Accuracy}
=
\frac{\text{맞힌 샘플 수}}
{\text{전체 샘플 수}}
$$

위 예에서는:

$$
\text{Accuracy} = \frac{3}{4} = 0.75
$$

코드에서는 다음 부분에 해당합니다.

```python
corrects += torch.sum(preds == labels.data)
epoch_acc = corrects.double() / len(train_loader.dataset)
```

수식으로 표현하면:

$$
\text{Epoch Accuracy}
=
\frac{
\sum_{i=1}^{N} \mathbb{1}(\hat y_i = y_i)
}{N}
$$

- \(N\): 전체 학습 샘플 수
- \(\hat y_i\): 모델의 예측 클래스
- \(y_i\): 실제 클래스
- \(\mathbb{1}\): 조건이 맞으면 1, 아니면 0

---

## 4. 실무에서 사용하는 방식

실무에서는 배치 크기를 고려하여 loss를 누적하는 방식이 더 안전합니다.

```python
def train(model, criterion, optimizer, train_loader, device):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for images, labels in train_loader:
        images = images.to(device, dtype=torch.float32)
        labels = labels.to(device, dtype=torch.long)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        batch_size = images.size(0)

        # loss는 배치 평균이므로 샘플 수를 곱해 합계로 변환
        total_loss += loss.item() * batch_size

        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += batch_size

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc
```

이 방식의 공식은 정확히 다음과 같습니다.

$$
L_{\text{epoch}}
=
\frac{
\sum_{i=1}^{N} L_i
}{N}
$$

$$
\text{Accuracy}
=
\frac{\text{정답 개수}}{N}
$$

### 이렇게 고친 이유

```python
total_loss += loss.item() * batch_size
```

`CrossEntropyLoss`의 기본 설정은 `reduction="mean"`이므로 `loss.item()`은 배치 평균입니다. 여기에 배치 크기를 곱하면 해당 배치의 loss 합계가 됩니다.

```python
total_correct += (preds == labels).sum().item()
```

`.item()`을 사용하면 GPU tensor를 Python 숫자로 변환합니다. 통계값을 계속 GPU tensor로 들고 있을 필요가 없습니다.

또한 다음 표현은 피하는 편이 좋습니다.

```python
labels.data
```

`.data`는 autograd와 관련된 문제를 유발할 수 있는 오래된 사용 방식입니다. 여기서는 그냥 `labels`를 사용하면 됩니다.

---

## 5. 검증 데이터도 함께 평가

실무에서는 학습 데이터의 accuracy만 보지 않고, 매 epoch마다 validation loss와 validation accuracy를 함께 확인합니다.

```python
def evaluate(model, criterion, data_loader, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device, dtype=torch.float32)
            labels = labels.to(device, dtype=torch.long)

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = images.size(0)

            total_loss += loss.item() * batch_size
            total_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            total_samples += batch_size

    return (
        total_loss / total_samples,
        total_correct / total_samples,
    )
```

검증에서는 다음 두 가지가 중요합니다.

```python
model.eval()
```

Dropout과 Batch Normalization을 평가 모드로 전환합니다.

```python
with torch.no_grad():
```

gradient 계산을 끄기 때문에 메모리 사용량과 연산량이 줄어듭니다.

---

## 6. 실무에서 accuracy만으로 부족한 경우

클래스가 균등하다면 accuracy가 직관적입니다. 하지만 클래스 불균형이 심하면 accuracy가 모델 성능을 과장할 수 있습니다.

예를 들어:

```text
정상 클래스: 990개
이상 클래스: 10개
```

모델이 모든 데이터를 정상이라고 예측해도 accuracy는:

$$
\frac{990}{1000}=99\%
$$

하지만 이상 데이터를 하나도 찾지 못한 모델입니다.

이런 경우 실무에서는 다음 지표를 함께 봅니다.

- Precision: 이상이라고 예측한 것 중 실제 이상인 비율
- Recall: 실제 이상 중 모델이 찾아낸 비율
- F1-score: Precision과 Recall의 조화 평균
- Confusion matrix: 클래스별 오분류 현황
- ROC-AUC 또는 PR-AUC
- 클래스별 accuracy 또는 recall

정리하면, 현재 코드의 계산 방식도 일반적인 구현이지만 실무에서는 배치 크기를 반영해 loss를 누적하고, 별도의 validation 데이터와 불균형에 적합한 평가 지표까지 함께 사용하는 것이 일반적입니다.

이 검증 함수는 모델의 가중치를 변경하지 않고 validation 데이터에 대한 평균 loss와 accuracy를 계산합니다.

학습 함수와 계산 공식은 거의 같지만, 검증에서는 역전파와 가중치 업데이트를 하지 않는다는 점이 핵심입니다.

## 전체 실행 흐름

```text
검증 모드로 변경
→ 검증 데이터 입력
→ 클래스별 로짓 계산
→ loss 계산
→ 가장 큰 로짓을 예측 클래스로 선택
→ 전체 loss와 정답 수 누적
→ epoch 평균 계산
```

## 1. 검증 모드 설정

```python
model.eval()
```

모델을 평가 모드로 변경합니다.

이 설정은 특히 다음 계층에 영향을 줍니다.

- `Dropout`: 일부 뉴런을 무작위로 끄지 않음
- `BatchNorm`: 현재 배치의 평균·분산 대신 학습 중 누적된 평균·분산 사용

현재 모델에는 `BatchNorm2d`가 있으므로 반드시 `model.eval()`이 필요합니다.

단, `model.eval()`은 gradient 계산 자체를 중지하지 않습니다. 모델 계층의 동작 모드만 변경합니다.

## 2. 통계값 초기화

```python
val_loss = 0.0
val_corrects = 0
```

- `val_loss`: 각 배치의 loss를 누적
- `val_corrects`: 모델이 정확히 예측한 샘플 수를 누적

## 3. Gradient 계산 중지

```python
with torch.no_grad():
```

검증에서는 모델을 학습시키지 않기 때문에 gradient가 필요하지 않습니다.

이를 사용하면:

- 연산 그래프를 만들지 않음
- 메모리 사용량 감소
- 검증 속도 향상
- 실수로 역전파에 사용하는 것을 방지

다만 `torch.no_grad()`도 모델을 평가 모드로 변경해 주지는 않습니다. 따라서 두 가지를 함께 사용합니다.

```python
model.eval()

with torch.no_grad():
    ...
```

## 4. 검증 배치 가져오기

```python
for images, labels in val_loader:
```

`val_loader`에서 이미지와 정답을 배치 단위로 가져옵니다.

배치 크기가 32라면 일반적으로 다음과 같은 형태입니다.

```text
images.shape = [32, 3, height, width]
labels.shape = [32]
```

예를 들어 클래스가 10개라면 각 label은 `0~9`의 정수입니다.

## 5. 데이터 타입 변환

```python
images, labels = images.float(), labels.long()
```

이미지는 모델 연산에 사용하기 위해 `float32`로 변환합니다.

```python
images.float()
```

`CrossEntropyLoss`의 정답은 클래스 인덱스여야 하므로 `int64`, 즉 `long`으로 변환합니다.

```python
labels.long()
```

예를 들면:

```python
labels = tensor([2, 0, 4, 1])
```

원-핫 벡터가 아니라 클래스 번호를 전달하는 것이 일반적입니다.

## 6. GPU 또는 CPU로 이동

```python
images, labels = images.to(device), labels.to(device)
```

모델과 데이터는 같은 장치에 있어야 합니다.

```python
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = model.to(device)
```

다음처럼 자료형과 장치 이동을 한 번에 처리할 수도 있습니다.

```python
images = images.to(device, dtype=torch.float32)
labels = labels.to(device, dtype=torch.long)
```

## 7. 모델 출력 계산

```python
outputs = model(images)
```

배치 크기가 \(B\), 클래스 개수가 \(C\)라면:

$$
\text{outputs.shape} = [B,C]
$$

예를 들어 이미지가 4장이고 클래스가 3개라면:

```python
outputs = tensor([
    [ 2.1,  0.4, -0.2],
    [ 0.1,  1.8,  0.5],
    [-0.4,  0.2,  2.5],
    [ 0.3,  1.1,  0.9],
])
```

각 행은 이미지 한 장에 대한 클래스별 로짓입니다. 아직 확률은 아닙니다.

## 8. 검증 loss 계산

```python
loss = criterion(outputs, labels)
```

`criterion`이 다음과 같다고 가정하겠습니다.

```python
criterion = nn.CrossEntropyLoss()
```

샘플 \(i\)의 클래스 \(j\)에 대한 로짓을 \(z_{i,j}\)라고 하면 softmax 확률은:

$$
p_{i,j}
=
\frac{e^{z_{i,j}}}
{\sum_{k=1}^{C}e^{z_{i,k}}}
$$

실제 정답 클래스가 \(y_i\)인 경우 샘플별 loss는:

$$
L_i=-\log(p_{i,y_i})
$$

기본 설정에서 배치 loss는 샘플별 loss의 평균입니다.

$$
L_{\text{batch}}
=
\frac{1}{B}\sum_{i=1}^{B}L_i
$$

`CrossEntropyLoss`가 내부적으로 softmax에 해당하는 연산까지 처리하므로 모델 출력에 별도로 softmax를 적용하지 않습니다.

## 9. Loss 누적

```python
val_loss += loss.item()
```

`loss.item()`은 배치의 평균 loss를 Python 숫자로 변환합니다.

모든 배치 크기가 같다면 문제가 없지만, 마지막 배치가 작을 경우 모든 배치에 같은 비중을 주는 문제가 있습니다.

현재 코드의 epoch loss는:

$$
L_{\text{epoch}}
=
\frac{1}{M}
\sum_{m=1}^{M} L_{\text{batch},m}
$$

여기서 \(M\)은 배치 개수입니다.

실무에서는 샘플 수를 반영하기 위해 다음처럼 계산하는 편이 정확합니다.

```python
batch_size = images.size(0)
val_loss += loss.item() * batch_size
```

마지막에는 전체 샘플 수로 나눕니다.

## 10. 예측 클래스 선택

```python
_, preds = torch.max(outputs, 1)
```

`dim=1` 방향, 즉 각 샘플의 클래스 점수 중 가장 큰 값을 찾습니다.

```python
preds = outputs.argmax(dim=1)
```

과 동일한 의미입니다.

예를 들어:

```python
outputs = tensor([
    [2.1, 0.4, -0.2],
    [0.1, 1.8,  0.5],
])
```

가장 큰 값의 인덱스는:

```python
preds = tensor([0, 1])
```

따라서 첫 번째 이미지는 클래스 0, 두 번째 이미지는 클래스 1로 예측한 것입니다.

softmax는 로짓의 크기 순서를 바꾸지 않으므로 클래스를 선택할 때는 softmax가 필요하지 않습니다.

```python
outputs.argmax(dim=1)
```

과

```python
F.softmax(outputs, dim=1).argmax(dim=1)
```

의 결과는 같습니다.

## 11. 정답 개수 누적

```python
val_corrects += torch.sum(preds == labels.data)
```

예를 들어:

```python
preds  = tensor([0, 1, 2, 1])
labels = tensor([0, 2, 2, 1])
```

비교 결과는:

```python
preds == labels
# tensor([True, False, True, True])
```

합계는 3입니다.

```python
torch.sum(preds == labels)
# tensor(3)
```

다만 `.data`는 권장되지 않는 오래된 사용 방식이므로 다음처럼 작성하는 것이 좋습니다.

```python
val_corrects += (preds == labels).sum().item()
```

## 12. Epoch loss 계산

```python
epoch_loss = val_loss / len(val_loader)
```

`len(val_loader)`는 검증 샘플 수가 아니라 배치 개수입니다.

예를 들어 검증 데이터가 100개이고 배치 크기가 32라면:

```python
len(val_loader.dataset)  # 100
len(val_loader)          # 4
```

현재 코드는 4개 배치의 평균 loss를 계산합니다.

## 13. Epoch accuracy 계산

```python
epoch_acc = (
    val_corrects.double()
    / len(val_loader.dataset)
)
```

정확도 공식은 다음과 같습니다.

$$
\text{Accuracy}
=
\frac{\text{정확히 예측한 샘플 수}}
{\text{전체 검증 샘플 수}}
$$

검증 데이터가 100개이고 85개를 맞혔다면:

$$
\text{Accuracy}
=
\frac{85}{100}
=
0.85
$$

즉 정확도는 85%입니다.

출력할 때는 다음처럼 표현할 수 있습니다.

```python
print(f"Val accuracy: {epoch_acc:.2%}")
```

## 실무에서 권장하는 구현

배치 크기가 서로 달라도 정확한 평균 loss가 계산되도록 다음처럼 구현할 수 있습니다.

```python
def validate(model, criterion, val_loader, device):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(
                device,
                dtype=torch.float32,
                non_blocking=True,
            )
            labels = labels.to(
                device,
                dtype=torch.long,
                non_blocking=True,
            )

            outputs = model(images)
            loss = criterion(outputs, labels)

            batch_size = images.size(0)

            # 배치 평균 loss를 배치 전체 loss로 변환
            total_loss += loss.item() * batch_size

            preds = outputs.argmax(dim=1)
            total_correct += (
                preds == labels
            ).sum().item()

            total_samples += batch_size

    epoch_loss = total_loss / total_samples
    epoch_acc = total_correct / total_samples

    return epoch_loss, epoch_acc
```

`torch.inference_mode()`는 검증처럼 gradient가 전혀 필요 없는 상황에서 `torch.no_grad()`보다 조금 더 강하게 autograd 관련 처리를 비활성화합니다.

`non_blocking=True`는 `DataLoader`에 `pin_memory=True`를 사용하고 GPU로 데이터를 옮길 때 성능에 도움이 될 수 있습니다.

```python
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    pin_memory=True,
)
```

검증 데이터는 순서를 섞을 필요가 없으므로 일반적으로 `shuffle=False`를 사용합니다.

## 학습 함수와 검증 함수의 차이

| 구분 | 학습 | 검증 |
|---|---|---|
| 모델 모드 | `model.train()` | `model.eval()` |
| Gradient 계산 | 사용 | 사용하지 않음 |
| `optimizer.zero_grad()` | 사용 | 사용하지 않음 |
| `loss.backward()` | 사용 | 사용하지 않음 |
| `optimizer.step()` | 사용 | 사용하지 않음 |
| 가중치 변경 | 변경됨 | 변경되지 않음 |
| Dropout | 활성화 | 비활성화 |
| BatchNorm | 현재 배치 통계 사용 | 학습 중 누적 통계 사용 |

검증 함수의 목적은 새로운 데이터에 대한 모델의 일반화 성능을 측정하는 것입니다. 그래서 검증 데이터는 모델 학습이나 하이퍼파라미터별 최종 성능 보고에 직접 섞이지 않도록 관리해야 합니다. 최종 성능은 검증 데이터가 아닌 별도의 test 데이터로 한 번 평가하는 것이 일반적입니다.

```python
from glob import glob
import os

def get_train_data(data_dir):
    img_path_list = []
    label_list = []

    img_path_list.extend(
        glob(os.path.join(data_dir, "*.jpg"))
    )

    img_path_list.sort(
        key=lambda x: int(
            os.path.splitext(os.path.basename(x))[0]
        )
    )

    label_list.extend(train_df["label"])

    return img_path_list, label_list


def get_test_data(data_dir):
    img_path_list = []

    img_path_list.extend(
        glob(os.path.join(data_dir, "*.jpg"))
    )

    img_path_list.sort(
        key=lambda x: int(
            os.path.splitext(os.path.basename(x))[0]
        )
    )

    return img_path_list


all_img_path, all_label = get_train_data("sign_train")
test_img_path = get_test_data("sign_test")

print(all_img_path[:10])
print(all_label[:10])
```

## custom dataset class and dataloader setting 

```python
import cv2
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# 사용자 정의 데이터셋 클래스
class CustomDataset(Dataset):
    def __init__(self, img_paths, labels=None, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        imgae = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        if self.labels is not None:
            label = self.labels[idx]
            return image, label
        else:
            return image
        
# 데이터셋 및 데이터로더 정의
train_img_paths, val_img_paths, train_labels, val_labels = train_test_split(all_img_path, all_label, test_size=0.2, random_state=42)

train_dataset = CustomDataset(train_img_paths, train_labels, transform=train_transform)
val_dataset = CustomDataset(val_img_paths, val_labels, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
```

## AlexNet 사전 학습 모델 불러오기 

```python
import torch
import torch.nn as nn
from torchvision import models

model = models.alexnet(pretrained=True)
print(f"[변경 전 모델]\n", model)
```

- `pretrained=True`로 설정하면 ImageNet 데이터 셋으로 학습된 가중치가 로드 된다. 
이는 모델이 이미 수백만 개의 이미지를 학습한 상태이기 때문에, 새로운 작업에 대해 더 빠르고 효율적으로 학습할 수 있다는 것을 의미 

```
[변경 전 모델]
 AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=4096, out_features=4096, bias=True)
    (5): ReLU(inplace=True)
    (6): Linear(in_features=4096, out_features=1000, bias=True)
  )
)
```

- Fully connected layer 에서 1000개의 클래스를 예측하도록 설계 되어 있음. 

- 1000개 보다 적은 11개의 클래스가 있기 때문에, 출력 계층을 우리 데이터셋의 클래스 수에 맞게 수정해야 한다. 

```python
num_classes = len(train_df['label'].unique())
model.classifier[6] = nn.Linear(model.classifier[6].in_features, 11)
print(f"[변경 후 모델]\n", model)
```

```
(avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
    (2): ReLU(inplace=True)
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=4096, out_features=4096, bias=True)
    (5): ReLU(inplace=True)
    (6): Linear(in_features=4096, out_features=11, bias=True)
```

```python
# <<-----여기에 작성해보세요----->>
def train(model, criterion, optimizer, train_loader):
    model.train()
    running_loss = 0.0
    corrects = 0
    
    for images, labels in tqdm(train_loader):
        images, labels = images.float(), labels.long()
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        
        _, preds = torch.max(outputs, 1)
        corrects += torch.sum(preds == labels.data)
    epoch_loss = running_loss / len(train_loader) #검증 데이터에 대한 손실
    epoch_acc = corrects.double() / len(train_loader.dataset) # 정확도
    return epoch_loss, epoch_acc
# 검증 함수 정의
def validate(model, criterion, val_loader):
    
    model.eval()
    val_loss = 0.0
    val_corrects = 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images, labels = images.float(), labels.long()  
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data)
            
    epoch_loss = val_loss / len(val_loader)
    epoch_acc = val_corrects.double() / len(val_loader.dataset)
    return epoch_loss, epoch_acc

```

```python
from tqdm import tqdm
import torch.optim as optim
import matplotlib.pyplot as plt

# 손실 함수 및 옵티마이저 정의
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

# 성능 결과를 저장할 리스트
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []

# 모델 학습 및 검증
num_epochs = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    train_loss, train_acc = train(model, criterion, optimizer, train_loader)
    val_loss, val_acc = validate(model, criterion, val_loader)
    train_losses.append(train_loss)
    train_accuracies.append(train_acc.item())
    val_losses.append(val_loss)
    val_accuracies.append(val_acc.item())
    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Train Loss: {train_loss:.4f}, "
          f"Train Accuracy: {train_acc:.4f}, "
          f"Validation Loss: {val_loss:.4f}, "
          f"Validation Accuracy: {val_acc:.4f}")

# 학습 및 검증 손실 그래프
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss')
plt.plot(range(1, num_epochs + 1), val_losses, label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train and Validation Loss')
plt.legend()
plt.show()

# 학습 및 검증 정확도 그래프
plt.figure(figsize=(10, 5))
plt.plot(range(1, num_epochs + 1), train_accuracies, label='Train Accuracy')
plt.plot(range(1, num_epochs + 1), val_accuracies, label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Train and Validation Accuracy')
plt.legend()
plt.show()
```

## 추론 

```python
from tqdm import tqdm

# 추론 함수 정의
def predict(model, test_loader, device):
    model.eval()
    model_pred = []
    with torch.no_grad():
        for img in tqdm(iter(test_loader)):
            img = img.to(device)
            pred_logit = model(img)
            pred_logit = pred_logit.argmax(dim=1, keepdim=True).squeeze(1) #가장 높은 확률의 클랫 선택
            model_pred.extend(pred_logit.tolist())
    return model_pred

# 테스트 데이터 로드
test_dataset = CustomDataset(test_img_path, labels=None, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

preds = predict(model, test_loader, device)
preds[:10]
```

## 라벨 값 변환

```python
preds = ["10-1" if x == 10 else x for x in preds]
preds = ["10-2" if x == 0 else x for x in preds ]

print(preds)
```

## 제출

```python
submission = pd.read_csv('sample_submission.csv')
submission['label'] = preds

submission.to_csv('submission.csv', index=False)
```

이 코드는 이미지 배치를 모델에 입력하고, 각 이미지에서 점수가 가장 높은 클래스를 골라 `model_pred` 리스트에 저장합니다.

```python
pred_logit = model(img)
pred_logit = pred_logit.argmax(
    dim=1,
    keepdim=True,
).squeeze(1)

model_pred.extend(pred_logit.tolist())
```

## 1. 모델의 출력

```python
pred_logit = model(img)
```

`img`가 여러 이미지로 구성된 배치라면 모델 출력 크기는 다음과 같습니다.

```text
[batch_size, num_classes]
```

예를 들어 배치 크기가 4이고 클래스가 3개라면:

```python
pred_logit = tensor([
    [ 2.1,  0.5, -0.4],
    [ 0.2,  3.7,  1.1],
    [-1.0,  0.4,  2.8],
    [ 0.8,  1.5,  1.2],
])
```

각 행은 이미지 한 장에 대한 클래스별 점수입니다.

```text
첫 번째 행 → 첫 번째 이미지
두 번째 행 → 두 번째 이미지
세 번째 행 → 세 번째 이미지
네 번째 행 → 네 번째 이미지
```

각 열은 클래스에 대응합니다.

```text
0번 열 → 클래스 0
1번 열 → 클래스 1
2번 열 → 클래스 2
```

중요한 점은 `model(img)`의 출력이 확률이 아니라 로짓이라는 것입니다.

```python
pred_logit = model(img)
```

따라서 변수 이름은 `pred_logits`처럼 복수형으로 표현하는 것이 더 자연스럽습니다.

## 2. 가장 큰 로짓의 클래스 선택

```python
pred_logit.argmax(dim=1, keepdim=True)
```

`argmax()`는 가장 큰 값 자체가 아니라 가장 큰 값의 인덱스를 반환합니다.

`dim=1`은 각 이미지에서 클래스 방향으로 가장 큰 값을 찾겠다는 뜻입니다.

예를 들어 첫 번째 이미지의 출력이 다음과 같다면:

```python
[2.1, 0.5, -0.4]
```

가장 큰 값은 `2.1`이고 그 위치는 인덱스 0입니다.

```python
argmax([2.1, 0.5, -0.4])
# 0
```

두 번째 이미지에서는:

```python
[0.2, 3.7, 1.1]
```

가장 큰 값은 `3.7`이고 인덱스는 1입니다.

```python
argmax([0.2, 3.7, 1.1])
# 1
```

전체 결과는 다음과 같습니다.

```python
tensor([
    [0],
    [1],
    [2],
    [1],
])
```

즉 모델의 예측 클래스는 다음과 같습니다.

```text
첫 번째 이미지 → 클래스 0
두 번째 이미지 → 클래스 1
세 번째 이미지 → 클래스 2
네 번째 이미지 → 클래스 1
```

## 3. `keepdim=True`

```python
argmax(dim=1, keepdim=True)
```

`keepdim=True`는 `argmax`로 줄어든 차원을 크기 1인 상태로 유지합니다.

입력 크기는:

```text
[4, 3]
```

이고 `keepdim=True`를 적용한 출력 크기는:

```text
[4, 1]
```

입니다.

```python
tensor([
    [0],
    [1],
    [2],
    [1],
])
```

반대로 `keepdim=False` 또는 기본값을 사용하면:

```python
pred_logits.argmax(dim=1)
```

결과 크기가 바로 다음처럼 됩니다.

```text
[4]
```

```python
tensor([0, 1, 2, 1])
```

## 4. `squeeze(1)`

```python
.squeeze(1)
```

크기가 1인 1번 차원을 제거합니다.

변환 전:

```text
shape = [4, 1]
```

```python
tensor([
    [0],
    [1],
    [2],
    [1],
])
```

변환 후:

```text
shape = [4]
```

```python
tensor([0, 1, 2, 1])
```

따라서 현재 코드는 차원을 유지했다가 바로 제거합니다.

```python
.argmax(dim=1, keepdim=True).squeeze(1)
```

결과적으로 `keepdim=True`와 `squeeze(1)`가 서로 반대 역할을 하기 때문에 다음처럼 간단히 쓸 수 있습니다.

```python
pred_class = pred_logits.argmax(dim=1)
```

## 5. Tensor를 Python 리스트로 변환

```python
pred_logit.tolist()
```

예측 클래스 tensor를 Python 리스트로 변환합니다.

```python
tensor([0, 1, 2, 1])
```

변환 결과:

```python
[0, 1, 2, 1]
```

GPU tensor라면 CPU로 명시적으로 옮긴 후 리스트로 변환하는 편이 코드 의도가 명확합니다.

```python
pred_class.cpu().tolist()
```

## 6. `extend()`로 결과 추가

```python
model_pred.extend(pred_logit.tolist())
```

현재 배치의 예측값을 `model_pred` 리스트에 하나씩 추가합니다.

예를 들어 기존 리스트가 다음과 같다면:

```python
model_pred = [2, 0]
```

현재 배치의 예측 결과가 다음과 같을 때:

```python
pred_class.tolist()
# [0, 1, 2, 1]
```

`extend()` 결과는 다음과 같습니다.

```python
model_pred
# [2, 0, 0, 1, 2, 1]
```

`append()`와는 결과가 다릅니다.

```python
model_pred.append([0, 1, 2, 1])
# [2, 0, [0, 1, 2, 1]]
```

```python
model_pred.extend([0, 1, 2, 1])
# [2, 0, 0, 1, 2, 1]
```

여러 배치의 예측을 하나의 평평한 리스트에 모으려면 `extend()`가 적합합니다.

## 더 간단하고 명확한 코드

```python
# 이미지 배치를 모델에 입력하여 클래스별 로짓을 계산한다.
# 출력 크기: [batch_size, num_classes]
pred_logits = model(img)

# 각 이미지에서 가장 큰 로짓의 인덱스를 예측 클래스로 선택한다.
# 출력 크기: [batch_size]
pred_classes = pred_logits.argmax(dim=1)

# GPU tensor를 CPU로 옮겨 Python 리스트로 변환한 뒤
# 전체 예측 결과 리스트에 추가한다.
model_pred.extend(pred_classes.cpu().tolist())
```

추론 전체 코드는 보통 다음처럼 작성합니다.

```python
model_pred = []

# BatchNorm과 Dropout을 평가 모드로 전환
model.eval()

# 추론에서는 gradient 계산이 필요하지 않음
with torch.inference_mode():
    for img in test_loader:
        # Dataset 반환 형태에 따라 필요한 부분만 사용
        if isinstance(img, (list, tuple)):
            img = img[0]

        img = img.to(device, dtype=torch.float32)

        # 클래스별 로짓
        pred_logits = model(img)

        # 로짓이 가장 큰 클래스 선택
        pred_classes = pred_logits.argmax(dim=1)

        # 전체 예측 결과에 추가
        model_pred.extend(pred_classes.cpu().tolist())
```

기존 주석도 다음처럼 수정하는 것이 정확합니다.

```python
# 각 이미지에서 가장 높은 로짓을 가진 클래스의 인덱스를 선택
pred_classes = pred_logits.argmax(dim=1)
```

“가장 높은 확률의 클래스”라고 이해해도 최종 클래스 선택 결과는 같지만, 모델이 직접 출력하는 값은 확률이 아니라 로짓입니다. Softmax는 로짓의 순서를 바꾸지 않기 때문에 예측 클래스만 필요하면 별도로 softmax를 계산할 필요가 없습니다.

`argmax`는 값들 중에서 가장 큰 값의 위치, 즉 인덱스를 반환하는 함수입니다.

`max`가 가장 큰 값 자체를 구한다면, `argmax`는 그 값이 몇 번째에 있는지를 구합니다.

## 간단한 예제

```python
import torch

x = torch.tensor([2.1, 5.7, 1.3])

print(torch.max(x))
# tensor(5.7000)

print(torch.argmax(x))
# tensor(1)
```

가장 큰 값은 `5.7`이고, 이 값은 인덱스 1에 있습니다.

```text
인덱스:  0    1    2
값:     2.1  5.7  1.3
             ↑
          가장 큰 값
```

따라서:

```python
x.max()
# tensor(5.7000)

x.argmax()
# tensor(1)
```

## CNN 분류에서의 `argmax`

분류 모델은 이미지 한 장마다 클래스별 로짓을 출력합니다.

```python
pred_logits = torch.tensor([
    [2.1, 0.5, 3.2],
    [0.3, 4.1, 1.7],
])
```

출력의 크기는 다음과 같습니다.

```text
[batch_size, num_classes] = [2, 3]
```

의미는 다음과 같습니다.

```text
             클래스 0  클래스 1  클래스 2
첫 번째 이미지   2.1       0.5       3.2
두 번째 이미지   0.3       4.1       1.7
```

각 이미지에서 가장 큰 값을 가진 클래스의 인덱스를 구합니다.

```python
pred_classes = pred_logits.argmax(dim=1)

print(pred_classes)
# tensor([2, 1])
```

결과를 해석하면:

```text
첫 번째 이미지 → 클래스 2
두 번째 이미지 → 클래스 1
```

## `dim=1`의 의미

```python
pred_logits.argmax(dim=1)
```

`dim=1`은 각 행 안에서 가장 큰 값의 위치를 찾으라는 의미입니다.

```text
각 행 = 이미지 한 장
각 열 = 클래스
```

따라서 클래스 방향으로 가장 큰 값을 찾으려면 `dim=1`을 사용합니다.

```python
pred_logits = torch.tensor([
    [2.1, 0.5, 3.2],
    [0.3, 4.1, 1.7],
])

pred_logits.argmax(dim=1)
# tensor([2, 1])
```

반대로 `dim=0`을 사용하면 각 열에서 가장 큰 값의 행 위치를 찾습니다.

```python
pred_logits.argmax(dim=0)
# tensor([0, 1, 0])
```

```text
클래스 0 열: [2.1, 0.3] → 첫 번째 행의 인덱스 0
클래스 1 열: [0.5, 4.1] → 두 번째 행의 인덱스 1
클래스 2 열: [3.2, 1.7] → 첫 번째 행의 인덱스 0
```

분류 문제에서는 일반적으로 이미지별 예측 클래스를 구해야 하므로 `dim=1`을 사용합니다.

## `torch.max()`와 비교

다음 두 코드는 비슷하지만 반환값이 다릅니다.

```python
values, indices = torch.max(pred_logits, dim=1)
```

결과:

```python
values
# tensor([3.2000, 4.1000])

indices
# tensor([2, 1])
```

`torch.max()`는 가장 큰 값과 인덱스를 모두 반환합니다.

예측 클래스의 인덱스만 필요하다면 다음이 더 간단합니다.

```python
pred_classes = pred_logits.argmax(dim=1)
```

정리하면:

```python
max_value = x.max()       # 가장 큰 값
max_index = x.argmax()    # 가장 큰 값의 위치
```

분류 모델에서는 가장 높은 로짓의 위치가 예측 클래스 번호이므로 `argmax(dim=1)`을 사용합니다.